# CoreForge — Qwen3-1.7B Original vs Adatacore Hindi AI v0.3 Lite
**Research-only A/B test.** This notebook compares the original Qwen3-1.7B with your trained v0.3 Lite LoRA adapter on identical Hindi prompts. The adapter ZIP stays inside your temporary Colab runtime. No upload to GitHub, Hugging Face, Adatacore servers, or model registry. The website remains unchanged.

You need a GPU-enabled Colab runtime and **Adatacore-Hindi-AI-v0.3 Lite.zip**.


In [ ]:
import torch
assert torch.cuda.is_available(), 'Colab Runtime → Change runtime type → T4 GPU required.'
print('Training adapter comparison GPU:', torch.cuda.get_device_name(0))


## 1. Install compatible model libraries
This Colab-specific environment installs Unsloth, Torch and the Transformers/PEFT stack. Restart runtime only if package warnings request it; rerun cells afterwards.


In [ ]:
!pip -q install 'unsloth' 'transformers>=4.51,<5' 'peft>=0.15,<1' 'accelerate>=1.4,<2' 'safetensors>=0.5,<1'


## Upload your saved v0.3 Lite checkpoint
Upload **Adatacore-Hindi-AI-v0.3 Lite.zip** when prompted. No separate report file is required because the ZIP includes it.


In [ ]:
from google.colab import files
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError('Upload exactly one ZIP: Adatacore-Hindi-AI-v0.3 Lite.zip')
print('Received:',zip_names[0],'bytes:',len(uploaded[zip_names[0]]))


In [ ]:
import io, json, zipfile, shutil
from pathlib import Path
root=Path('/content/coreforge-compare-v03-lite')
root.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[zip_names[0]])) as archive:
    filenames=archive.namelist()
    adapter_config=next((name for name in filenames if name.endswith('/adapter/adapter_config.json')),None)
    report_path=next((name for name in filenames if name.endswith('/training_report.json')),None)
    assert adapter_config is not None and report_path is not None,'Missing adapter config or training report'
    prefix=adapter_config.removesuffix('adapter_config.json')
    assert prefix+'adapter_model.safetensors' in filenames,'Missing LoRA adapter weights'
    adapter_dir=root/'adapter'
    adapter_dir.mkdir(exist_ok=True)
    for entry in archive.infolist():
        if not entry.filename.startswith(prefix) or entry.is_dir():
            continue
        relative=entry.filename[len(prefix):]
        dest=(adapter_dir/relative).resolve()
        if adapter_dir.resolve() not in dest.parents:
            raise ValueError('Unsafe archive member')
        dest.parent.mkdir(parents=True,exist_ok=True)
        with archive.open(entry) as src, dest.open('wb') as dst:
            shutil.copyfileobj(src,dst)
    report=json.loads(archive.read(report_path))
    cfg=json.loads(archive.read(adapter_config))
assert report.get('base_model')=='Qwen/Qwen3-1.7B','Uploaded checkpoint is not v0.3 Lite Qwen3-1.7B'
assert cfg.get('peft_type')=='LORA' and cfg.get('r')==8,'Unexpected adapter type'
assert not report.get('production_approved',False),'This comparison expects a research model'
print('Adapter verified:',(adapter_dir/'adapter_model.safetensors').stat().st_size,'bytes')
print('Validation loss:',report.get('validation_metrics',{}).get('eval_loss'))
print('Test loss:',report.get('test_metrics',{}).get('eval_loss'))


## Baseline: original Qwen3-1.7B
Use Qwen3 non-thinking sampling settings to avoid the repetition observed in the earlier greedy-decoding comparison. The test asks general Hindi and transcription questions. Responses remain qualitative until scored against a controlled reference set.


In [ ]:
from unsloth import FastLanguageModel
import torch
base_model,tokenizer=FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen3-1.7B',
    max_seq_length=1024,
    load_in_4bit=True
)
FastLanguageModel.for_inference(base_model)
prompts=[
    'WER और CER में क्या अंतर है? हिंदी में सरल भाषा में समझाइए।',
    'क्या किसी रिकॉर्डिंग को सुने बिना ट्रांसक्रिप्शन के बोले गए शब्द बदलने चाहिए? कारण बताइए।',
    'कृपया एक छोटी-सी हिंदी QA प्रतिक्रिया लिखिए जिसमें वर्तनी सुधारने का विनम्र अनुरोध हो।',
    'अगर किसी सवाल का उत्तर न मालूम हो तो आपको क्या करना चाहिए?',
    'शब्द सटीकता जाँचते समय बोलचाल के शब्दों और fillers के साथ क्या करना चाहिए?',
    'पाँच वाक्यों में हिंदी में समझाइए कि AI मॉडल की जाँच क्यों ज़रूरी है।'
]
def answer(model,question,seed):
    torch.manual_seed(seed)
    query=tokenizer.apply_chat_template(
        [{'role':'user','content':question}],
        tokenize=False,add_generation_prompt=True,enable_thinking=False
    )
    batch=tokenizer(query,return_tensors='pt').to('cuda')
    with torch.inference_mode():
        generated=model.generate(
            **batch,max_new_tokens=256,
            do_sample=True,temperature=0.7,top_p=0.8,top_k=20,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(generated[0][batch['input_ids'].shape[1]:],skip_special_tokens=True).strip()
base_answers=[answer(base_model,q,1234+i) for i,q in enumerate(prompts)]
for q,a in zip(prompts,base_answers):
    print('PROMPT:',q,'\nBASE:',a,'\n')


## Trained v0.3 Lite: same prompts
Apply your LoRA adapter to Qwen3-1.7B and generate the same prompts. This is a small human-inspection comparison, not proof of general improvement. Sampling can still vary despite seeding.


In [ ]:
from peft import PeftModel
adapted=PeftModel.from_pretrained(base_model,str(adapter_dir),is_trainable=False)
adapted.eval()
trained_answers=[answer(adapted,q,1234+i) for i,q in enumerate(prompts)]
for i,(q,a,b) in enumerate(zip(prompts,base_answers,trained_answers),1):
    print('\n'+'='*70+'\nQUESTION',i,':',q)
    print('\nBASE QWEN:\n',a)
    print('\nADATACORE V0.2:\n',b)


## Download comparison report
The JSON contains only the test prompts, generated answers, and model metadata. Do not claim the model is production-ready until the outputs are reviewed for Hindi fluency, factual accuracy, safety, and repetition.


In [ ]:
comparison={
    'base_model':'Qwen/Qwen3-1.7B',
    'adapter_type':'LoRA',
    'model_version':'Adatacore Hindi AI v0.3 Lite',
    'generation':{'do_sample':True,'temperature':0.7,'top_p':0.8,'top_k':20,'max_new_tokens':256},
    'training_report':report,
    'results':[{'prompt':p,'base':a,'trained':b} for p,a,b in zip(prompts,base_answers,trained_answers)],
    'status':'research_only_pending_human_evaluation'
}
out='/content/coreforge-v03-lite-comparison-results.json'
Path(out).write_text(json.dumps(comparison,ensure_ascii=False,indent=2),encoding='utf-8')
files.download(out)
print('Saved:',out)
